In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

In [1]:
import json
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 1. Full Text Extraction

def scrape_article_text(url):
    """Scrapes paragraph text from a given URL."""
    try:
        # Timeout is crucial so dead links do not hang the script
        response = requests.get(url, timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
        if response.status_code != 200:
            return ""
        
        soup = BeautifulSoup(response.content, 'html.parser')
        # Extract text from all paragraph tags
        paragraphs = soup.find_all('p')
        full_text = " ".join([p.get_text(strip=True) for p in paragraphs])
        return full_text
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")
        return ""


# Main Execution Block

if __name__ == "__main__":
    input_file = "nvidia_clean_intelligence.jsonl"
    
    print("Step 1: Extracting full text from URLs")
    raw_documents = []
    
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            url = data.get("url")
            
            # Skip Hacker News API links without external URLs
            if not url:
                continue
                
            body_text = scrape_article_text(url)
            
            # Only keep documents where successfully extracted substantial text
            if len(body_text) > 200:
                # Create LangChain Document objects with metadata attached
                doc = Document(
                    page_content=body_text,
                    metadata={
                        "source": data.get("source_name", "Unknown"),
                        "title": data.get("title", ""),
                        "date": data.get("date", ""),
                        "url": url
                    }
                )
                raw_documents.append(doc)

    print(f"Successfully extracted text for {len(raw_documents)} documents.")


    # 2. Chunking the Text

    print("Step 2: Splitting text into chunks")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    chunked_docs = text_splitter.split_documents(raw_documents)
    print(f"Created {len(chunked_docs)} text chunks.")

    # 3. Generating Embeddings and Indexing
    
    print("Step 3: Generating embeddings and indexing into ChromaDB...")
    
    # Using the exact open-source embedding model recommended in the requirements
    model_name = "BAAI/bge-base-en-v1.5"
    model_kwargs = {'device': 'cuda'}
    encode_kwargs = {'normalize_embeddings': True}
    
    hf_embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    
    # Store locally in a directory called 'chroma_db'
    persist_directory = "./chroma_db"
    
    vectorstore = Chroma.from_documents(
        documents=chunked_docs,
        embedding=hf_embeddings,
        persist_directory=persist_directory
    )
    
    print(f"\n Knowledge Repository successfully built at {persist_directory}")

/home/jovyan/vault/AI_CEO_NLP/ai_ceo_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_235/390840872.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Step 1: Extracting full text from URLs
Successfully extracted text for 90 documents.
Step 2: Splitting text into chunks
Created 941 text chunks.
Step 3: Generating embeddings and indexing into ChromaDB...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7492.52it/s]



 Knowledge Repository successfully built at ./chroma_db
